# 05 Materials Project Screening

This notebook inspects the screening table, demonstrates the filtering logic for rescued candidates, and documents the optional CIF acquisition workflow.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(repo_root)


In [ ]:
import os
import pandas as pd

screen = pd.read_csv(repo_root / 'data' / 'mp_screening_results.csv')
print(screen.shape)
screen.head()


In [ ]:
required_cols = ['pretty_formula', 'band_gap', 'Predicted_Is_Metal', 'Final_Predicted_Bandgap', 'formation_energy_per_atom']
missing = [c for c in required_cols if c not in screen.columns]
print('missing columns:', missing)


In [ ]:
rescued = screen[(screen['is_metal_gga'] == 1) & (screen['Predicted_Is_Metal'] == 0)]
pb_free = rescued[~rescued['pretty_formula'].str.contains('Pb', na=False)]
stable = pb_free[pb_free['formation_energy_per_atom'] < 0].copy()
stable['sq_diff'] = (stable['Final_Predicted_Bandgap'] - 1.34).abs()
top_candidates = stable.sort_values('sq_diff')[['pretty_formula', 'band_gap', 'Final_Predicted_Bandgap', 'formation_energy_per_atom']].head(10)
print({'rescued': len(rescued), 'pb_free': len(pb_free), 'stable': len(stable)})
top_candidates


In [ ]:
print('MP_API_KEY set:', bool(os.getenv('MP_API_KEY')))
print('available CIF files:', len(list((repo_root / 'data' / 'cifs').glob('*.cif'))))


In [ ]:
from src.screening import fetch_cifs
print('Run fetch_cifs() only after setting MP_API_KEY in the environment.')
